In [ ]:
import pandas as pd
import numpy as np
import torch
import pyarrow.parquet as pq
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score
from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
# Load embeddings based on sentence-transformers model
schizophrenia_embeddings = np.load('../data/schizophrenia_embeddings.npz')['arr_0']

# Load the corresponding dataset
data = pq.read_table('../data/data_activelearning.parquet')
data = data.to_pandas()
schizophrenia_dataset = data.query("id_subcorpus == 'psyB'").reset_index() # Limit to schizophrenia subcorpus

# Load initialization dataset from csv 
schizophrenia_init_dataset = pd.read_csv('../data/schizophrenia_init_labeled.csv')

# Load test dataset from csv
schizophrenia_test_dataset = pd.read_csv('../data/schizophrenia_test_labeled.csv', delimiter = ';')

In [ ]:
# We need the ids
schizophrenia_init_ids = schizophrenia_init_dataset['id'].tolist()
schizophrenia_init_indices = schizophrenia_dataset.query('id in @schizophrenia_init_ids').index.to_numpy()
schizophrenia_init_embeddings = schizophrenia_embeddings[schizophrenia_init_indices]

# We need the ids of the test set too
schizophrenia_test_ids = schizophrenia_test_dataset['id'].tolist()
schizophrenia_test_indices = schizophrenia_dataset.query('id in @schizophrenia_test_ids').index.to_numpy()
test_init_indices = np.concatenate([schizophrenia_test_indices, schizophrenia_init_indices])

# Create boolean mask
mask = np.ones(len(schizophrenia_embeddings), dtype=bool)
mask[test_init_indices] = False

# Now we need to retain only the unlabeled pool of embeddings, so embeddings - init - test
schizophrenia_unlabeled_embeddings = schizophrenia_embeddings[mask]

# However, we need the positive and negative initial embeddings seperately for the method to work
schizophrenia_init_labels = schizophrenia_init_dataset['label_binary'].to_numpy()

# Create masks for positive and negative samples
pos_mask = schizophrenia_init_labels == 1
neg_mask = schizophrenia_init_labels == 0

# Filter IDs
schizophrenia_init_ids_pos = schizophrenia_init_dataset.loc[pos_mask, 'id'].tolist()
schizophrenia_init_ids_neg = schizophrenia_init_dataset.loc[neg_mask, 'id'].tolist()

# Filter indices
schizophrenia_init_indices_pos = schizophrenia_init_indices[pos_mask]
schizophrenia_init_indices_neg = schizophrenia_init_indices[neg_mask]

# Filter embeddings
schizophrenia_init_embeddings_pos = schizophrenia_init_embeddings[pos_mask]
schizophrenia_init_embeddings_neg = schizophrenia_init_embeddings[neg_mask]


In [11]:
def bootstrap_cosine(init_pos, init_neg, unlabeled, init_pos_idx, init_neg_idx, n_rounds=5, sample_size=100, balance=True, save_csv_path=None):
    """
    Bootstraps unlabeled embeddings into init_pos/init_neg by pseudo-labeling with cosine similarity.

    init_pos, init_neg: np.ndarray [n_pos, d], [n_neg, d]
    unlabeled: np.ndarray [n_unlabeled, d]
    n_rounds: number of bootstrap iterations
    sample_size: number of samples to add per round
    balance: if True, split roughly half pos/neg per round based on pseudo-labels

    Returns: (updated_init_pos, updated_init_neg, remaining_unlabeled)
    """
    init_pos = np.array(init_pos, copy=True)
    init_neg = np.array(init_neg, copy=True)
    unlabeled = np.array(unlabeled, copy=True)

    history = []
    all_chosen_pos = []  # Track all chosen positive indices
    all_chosen_neg = []  # Track all chosen negative indices

    for r in range(n_rounds):

        # Similarities to current init sets

        sim_pos = cosine_similarity(init_pos, unlabeled)
        max_sim_pos = sim_pos.max(axis=0)

        sim_neg = cosine_similarity(init_neg, unlabeled)
        max_sim_neg = sim_neg.max(axis=0)


        # Pseudo labels and confidence
        pseudo_labels = (max_sim_pos > max_sim_neg).astype(int)
        confidence = np.abs(max_sim_pos - max_sim_neg)

        # Sort by confidence desc
        sorted_idx = np.argsort(-confidence)

        if balance:
            pos_take = sample_size // 2
            neg_take = sample_size - pos_take

            sorted_labels = pseudo_labels[sorted_idx]
            pos_sorted = sorted_idx[sorted_labels == 1][:pos_take]
            neg_sorted = sorted_idx[sorted_labels == 0][:neg_take]

            chosen_idx = np.concatenate([pos_sorted, neg_sorted])
        else:
            chosen_idx = sorted_idx[:sample_size]
            pos_sorted = chosen_idx[pseudo_labels[chosen_idx] == 1]
            neg_sorted = chosen_idx[pseudo_labels[chosen_idx] == 0]

        # Select embeddings to add
        add_pos = unlabeled[pos_sorted]
        add_neg = unlabeled[neg_sorted]

        # Update init sets
        if add_pos.size:
            init_pos = add_pos if init_pos.size == 0 else np.vstack([init_pos, add_pos])
        if add_neg.size:
            init_neg = add_neg if init_neg.size == 0 else np.vstack([init_neg, add_neg])

                # Track cumulative chosen indices
        all_chosen_pos.extend(pos_sorted.tolist())
        all_chosen_neg.extend(neg_sorted.tolist())

        # Remove chosen from unlabeled
        keep_mask = np.ones(unlabeled.shape[0], dtype=bool)
        keep_mask[chosen_idx] = False
        unlabeled = unlabeled[keep_mask]  # keep only those not chosen -> invert

        training_set_idx = init_pos_idx + init_neg_idx + all_chosen_pos + all_chosen_neg
        training_set_labels = [1] * len(init_pos_idx) + [0] * len(init_neg_idx) + [1] * len(all_chosen_pos) + [0] * len(all_chosen_neg)

        history.append({
            'round': r + 1,
            'pos_added': int(add_pos.shape[0]),
            'neg_added': int(add_neg.shape[0]),
            'chosen_idx': chosen_idx.tolist(),
            'pos_idx': pos_sorted.tolist(),
            'neg_idx': neg_sorted.tolist(),
            'training_set_idx': training_set_idx,
            'training_set_labels': training_set_labels,
            'total_training_size': len(training_set_idx),
            'remaining_unlabeled': int(unlabeled.shape[0])
        })

    if save_csv_path:
        pd.DataFrame(history).to_csv(save_csv_path, index=False)

    return init_pos, init_neg, unlabeled, history

In [16]:
updated_pos, updated_neg, remaining_unlabeled, history = bootstrap_cosine(
    schizophrenia_init_embeddings_pos,
    schizophrenia_init_embeddings_neg,
    schizophrenia_unlabeled_embeddings,
    init_pos_idx=schizophrenia_init_indices_pos.tolist(),
    init_neg_idx=schizophrenia_init_indices_neg.tolist(),
    n_rounds=3,
    sample_size=100
)

In [ ]:
# Create test dataset
test_labels = schizophrenia_test_dataset['label_new'].tolist()
test_texts = schizophrenia_test_dataset['text'].tolist()
test_dataset = Dataset.from_dict({
    'text': test_texts,
    'label': test_labels
})

In [ ]:
# Run the initial training and evaluation before bootstrapping
training_args = TrainingArguments(
    seed=1234,
    batch_size=16,
    body_learning_rate=2e-5,
    max_length=128,
    num_iterations=20,
    num_epochs=1
    
)

# Train on seed set only (baseline)
seed_train_dataset = Dataset.from_dict({
    'text': schizophrenia_init_dataset['text'].tolist(),
    'label': schizophrenia_init_dataset['label_binary'].tolist()
})

seed_model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
seed_trainer = Trainer(
    model=seed_model,
    train_dataset=seed_train_dataset,
    eval_dataset=test_dataset,
    args=training_args
)

seed_trainer.train()

seed_predictions = seed_model.predict(test_texts)
seed_results = {
    'round': 0,
    'number_labeled': len(seed_train_dataset),
    'disease': 'schizophrenia',
    'f1_minority': f1_score(test_labels, seed_predictions, pos_label=0),
    'f1_macro': f1_score(test_labels, seed_predictions, average='macro')
}
results = [seed_results]

In [ ]:
# Train and evaluate on each round of bootstrapped data

for round_idx, round_history in enumerate(history):
    train_dataset = Dataset.from_dict({
        'text': schizophrenia_dataset.iloc[round_history['training_set_idx']]['text'].tolist(),
        'label': round_history['training_set_labels']
    })
    
    model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    if torch.cuda.is_available():
        model.model_body.to("cuda")
    training_args = TrainingArguments(
        seed=42,
        batch_size=16,
        body_learning_rate=2e-5,
        max_length=128,
        num_iterations=20,
        num_epochs=1
        
    )
    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        args=training_args
    )
    trainer.train()
    
    predictions = model.predict(test_texts)
    results.append({
        'round': round_idx + 1,
        'number_labeled': len(train_dataset),
        'disease': 'schizophrenia',
        'f1_minority': f1_score(test_labels, predictions, pos_label=0),
        'f1_macro': f1_score(test_labels, predictions, average='macro')
    })

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('../data/schizophrenia_bootstrap_results.csv')